In [1]:
%matplotlib widget
%matplotlib widget
import os
from pathlib import Path
import time
import torch
import numpy as np
import math
import gc
from functools import partial
from dataset_alt import Dataset, load_dataframes_from_folder, reverse_normalization
from torch.utils.data import DataLoader
from transformer_zerostep import GPTConfig, GPT, warmup_cosine_lr, GPT_chopped
import argparse
import warnings
import matplotlib.pyplot as plt
import onnxruntime as rt
import onnx
import copy
from collections import OrderedDict
import pickle as pkl


# set figure parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams['axes.labelsize']=14
plt.rcParams['xtick.labelsize']=11
plt.rcParams['ytick.labelsize']=11
plt.rcParams['axes.grid']=True
plt.rcParams['axes.xmargin']=0

In [2]:
# Overall settings
#if this doesn't work istg
out_dir = "out"

model_name = "new_delay_h10_10k.pt"
# model_name = "model_high_speed.pt"

current_path = os.getcwd().split("in-context-bldc")[0]
data_path = os.path.join(current_path,"in-context-bldc", "data")

folder = "simulated/50_percent_control/validation"
# folder = "CL_experiments_double_sensor_control/test/inertia13"
folder_path = os.path.join(data_path, folder)

# Compute settings
cuda_device = "cuda:0"
no_cuda = True
threads = 10
compile = False

# Configure compute
torch.set_num_threads(threads) 
use_cuda = not no_cuda and torch.cuda.is_available()
device_name  = cuda_device if use_cuda else "cpu"
device = torch.device(device_name)
device_type = 'cuda' if 'cuda' in device_name else 'cpu' # for later use in torch.autocast
torch.set_float32_matmul_precision("high")
print(torch.cuda.is_available())
# Create out dir
out_dir = Path(out_dir)
exp_data = torch.load(out_dir/model_name, map_location=device, weights_only=False)
seq_len = exp_data["cfg"].seq_len
nx = exp_data["cfg"].nx
exp_data["iter_num"]
print(seq_len)
print(exp_data["iter_num"])
print(exp_data['best_val_loss'])
print(exp_data["cfg"])
print(exp_data["cfg"].lr)
print(exp_data["train_time"]/3600)
print(exp_data["model_args"])

True
10
9888
0.0005529707050300203
Namespace(model_dir='out', out_file='new_delay_h10', in_file='new_delay_h10', init_from='scratch', seed=42, log_wandb=False, nx=4, nu=8, ny=1, seq_len=10, mag_range=(0.5, 0.97), phase_range=(0.0, 1.5707963267948966), fixed_system=False, n_layer=8, n_head=4, n_embd=16, dropout=0, bias=False, batch_size=128, max_iters=30000, warmup_iters=5000, lr=1e-05, weight_decay=0.0, eval_interval=10, eval_iters=10, fixed_lr=False, threads=16, no_cuda=False, cuda_device='cuda:0', compile=False, beta1=0.9, beta2=0.95, block_size=10, lr_decay_iters=30000, min_lr=1.0000000000000002e-06, decay_lr=True, eval_batch_size=128)
1e-05
3.96572324819035
{'n_layer': 8, 'n_head': 4, 'n_embd': 16, 'n_x': 4, 'n_y': 1, 'n_u': 8, 'block_size': 10, 'bias': False, 'dropout': 0}


In [3]:
# generate the model
model_args = exp_data["model_args"]
gptconf = GPTConfig(**model_args)
model = GPT(gptconf).to(device)


state_dict = exp_data["model"]

keys_raw = state_dict.keys()
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    if k.startswith('module.'):
        state_dict[k[7:]] = v
        state_dict.pop(k)

keys = state_dict.keys()


model.load_state_dict(state_dict)

number of parameters: 0.03M


<All keys matched successfully>

In [4]:
print(len(keys_raw))
print(len(keys))

54
54


In [5]:
model_name[:-3]

'new_delay_h10_10k'

In [8]:
test_input = torch.rand(5,gptconf.block_size,gptconf.n_u)
print(test_input.size())
test_output = model(test_input)
print(test_output.size())
# print(test_output)
gptconf_ord_dict = OrderedDict(gptconf.__dict__)
state_dict.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input, 'test_output': test_output}
state_dict.update(in_out_dict)

state_dict.keys()


data_to_save = state_dict
name_to_save = model_name[:-3] + '_weights.pkl'

with open(name_to_save,'wb') as f:
    pkl.dump(data_to_save, f)



torch.Size([5, 10, 8])
torch.Size([5, 10, 1])


In [9]:
print(test_output)

tensor([[[ 0.8516],
         [-0.0671],
         [ 0.6681],
         [ 0.9162],
         [ 0.2188],
         [ 0.8987],
         [ 0.9264],
         [-0.2356],
         [ 0.7144],
         [ 0.5153]],

        [[-0.2003],
         [ 0.9492],
         [ 0.3143],
         [-0.1065],
         [ 0.9071],
         [ 0.6492],
         [ 0.6324],
         [-0.3331],
         [ 0.7147],
         [ 0.4783]],

        [[-0.5276],
         [ 0.9262],
         [ 0.9047],
         [ 0.0743],
         [-0.1410],
         [ 0.9978],
         [-0.1195],
         [ 0.8761],
         [ 0.9242],
         [-0.0965]],

        [[ 0.8371],
         [ 0.2533],
         [-0.1374],
         [-0.3819],
         [ 0.2718],
         [-0.0439],
         [-0.0251],
         [ 0.9954],
         [ 0.7999],
         [ 0.1569]],

        [[ 0.0028],
         [ 0.8709],
         [ 0.8298],
         [ 0.3005],
         [ 0.3383],
         [ 0.3299],
         [ 0.3849],
         [ 0.9017],
         [ 0.7680],
         [ 0

In [7]:
model_chopped = GPT_chopped(gptconf)
test_input_2 = torch.rand(1,10,8)
test_output_2 = model_chopped(test_input_2)
state_dict2 = model_chopped.state_dict()

gptconf_ord_dict = OrderedDict(gptconf.__dict__)
state_dict2.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input_2, 'test_output': test_output_2}
state_dict2.update(in_out_dict)

state_dict2.keys()


data_to_save = state_dict2
name_to_save = 'test_chopped_weights.pkl'

with open(name_to_save,'wb') as f:
    pkl.dump(data_to_save, f)

number of parameters: 0.00M
